# 02_Bronze_AutoLoader
This notebook sets up Auto Loader streams to ingest CSV files for customers, products, and orders into the Bronze layer using Unity Catalog.
Each stream writes to a Delta table in the `ecommerce_catalog.bronze` schema with checkpointing.
Run this notebook once to start the streaming queries.
*Note*: In a real production environment you would keep the notebook running or schedule it as a job.


In [ ]:
from pyspark.sql import functions as F

CATALOG = "ecommerce_catalog"
BASE_PATH = "abfss://mycontainer@myaccount.dfs.core.windows.net/ecommerce/source/"  # Change to your ADLS path
SCHEMA_PATH = "/tmp/ecommerce/schema/"  # Local temp for schema inference
CHECKPOINT_ROOT = "/tmp/ecommerce/checkpoints/bronze/"

# ------------------- Customers -------------------
customers_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH + "customers")
        .load(BASE_PATH + "customers/")
)
customers_bronze = (customers_stream
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name()))

(customers_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_ROOT + "customers")
    .toTable(f"{CATALOG}.bronze.customers"))

# ------------------- Products -------------------
products_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH + "products")
        .load(BASE_PATH + "products/")
)
products_bronze = (products_stream
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name()))

(products_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_ROOT + "products")
    .toTable(f"{CATALOG}.bronze.products"))

# ------------------- Orders -------------------
orders_stream = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaLocation", SCHEMA_PATH + "orders")
        .load(BASE_PATH + "orders/")
)
orders_bronze = (orders_stream
    .withColumn("_ingestion_timestamp", F.current_timestamp())
    .withColumn("_source_file", F.input_file_name()))

(orders_bronze.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", CHECKPOINT_ROOT + "orders")
    .toTable(f"{CATALOG}.bronze.orders"))

print("Bronze Auto Loader streams started.")
